# 📙 หน่วยที่ 3 — Tools & Agents

```
main()                     ← หลัก
├── setup_llm()            ← รอง
├── create_tools()         ← รอง
├── demo_bind_tools()      ← รอง
├── create_agent()         ← รอง
└── run_agent()            ← รอง
```

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')
print(f'✅ Model: {OPENAI_MODEL}')

## 🔧 ฟังก์ชันรอง 1: `setup_llm()`

In [ ]:
def setup_llm():
    return ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0.0)

# เรียกใช้
llm = setup_llm()
print('✅ LLM พร้อม')

## 🛠️ ฟังก์ชันรอง 2: `create_tools()`

`@tool` = สอนสกิลใหม่ให้ AI | **Docstring สำคัญมาก** → AI อ่านเพื่อตัดสินใจ

In [ ]:
def create_tools():
    @tool
    def get_gold_price() -> str:
        """ดึงราคาทองคำ XAUUSD ปัจจุบัน"""
        return 'ราคาทองคำ: 2675.50 USD'

    @tool
    def calculate_profit(entry_price: float, exit_price: float, lot_size: float = 1.0) -> str:
        """คำนวณกำไรขาดทุนเทรดทองคำ
        Args: entry_price: ราคาเข้า | exit_price: ราคาออก | lot_size: ขนาด lot"""
        profit = (exit_price - entry_price) * 100 * lot_size
        return f'{"กำไร" if profit > 0 else "ขาดทุน"}: ${profit:,.2f}'

    return [get_gold_price, calculate_profit]

# เรียกใช้
tools = create_tools()
print(f'✅ {len(tools)} tools: {[t.name for t in tools]}')

## 🧠 ฟังก์ชันรอง 3: `demo_bind_tools()`

`bind_tools()` = บอก LLM ว่ามี Tools ให้ใช้ → AI เลือกเรียกเอง

In [ ]:
def demo_bind_tools(llm, tools):
    llm_t = llm.bind_tools(tools)
    r1 = llm_t.invoke('สวัสดี')
    print(f'ถามทั่วไป → tool_calls: {bool(r1.tool_calls)}')
    r2 = llm_t.invoke('ราคาทองตอนนี้?')
    print(f'ถามราคาทอง → tool_calls: {r2.tool_calls}')
    print('💡 AI เลือกเรียก Tool เอง!')

# เรียกใช้
demo_bind_tools(llm, tools)

## 🤖 ฟังก์ชันรอง 4: `create_agent()`

Agent = AI คิด → เรียก Tool → ดูผล → คิดต่อ → ตอบ (วนลูปอัตโนมัติ)

In [ ]:
def create_agent(llm, tools):
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'ผู้ช่วย AI ด้านเทรด ใช้ Tools ได้ ตอบไทย'),
        ('human', '{input}'),
        MessagesPlaceholder(variable_name='agent_scratchpad'),
    ])
    agent = create_tool_calling_agent(llm, tools, prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=True)

# เรียกใช้
executor = create_agent(llm, tools)
print('✅ Agent พร้อม!')

## 🎯 ฟังก์ชันรอง 5: `run_agent()`

สั่งงานที่ต้องใช้ **2 Tools ต่อเนื่อง** → Agent จัดการเอง

In [ ]:
def run_agent(executor, question):
    result = executor.invoke({'input': question})
    print(f'\n📥 คำตอบ: {result["output"]}')

# เรียกใช้
run_agent(executor, 'เช็คราคาทอง ถ้าซื้อ 2 lot แล้วขาย 2685 ได้กำไรเท่าไหร่?')

---
## ★ ฟังก์ชันหลัก `main()` — รันทุกอย่างรวมกัน

In [ ]:
def main():
    llm = setup_llm()                              # รอง 1
    tools = create_tools()                         # รอง 2
    demo_bind_tools(llm, tools)                    # รอง 3
    executor = create_agent(llm, tools)            # รอง 4
    run_agent(executor,                            # รอง 5
        'เช็คราคาทอง ถ้าซื้อ 2 lot แล้วขาย 2685 ได้กำไรเท่าไหร่?')
    print('\n✅ จบหน่วยที่ 3!')

# เรียกใช้
main()